# AVA v3-Code — resumable training notebook (Colab / Kaggle / laptop)

**One loop, three platforms, preemption-safe.** Every platform is a disposable worker.
The canonical state lives in a Hugging Face Hub repo. Any session can die; re-run this
notebook anywhere and it resumes from the last push within one sync interval.

Pattern (from `scripts/checkpoint_sync.py`):
- checkpoint = **trainable params + optimizer + RNG + step**, pushed every `SYNC_MINUTES`
- `LATEST.json` is uploaded **last** → a torn upload never corrupts resume
- data order is deterministic from `seed + step` → we fast-forward the stream on resume
- each session is **time-boxed** to `SHARD_MINUTES` (< platform limit), then force-saves and exits

Phases (CODE_PIVOT.md §8): **C1** donor baseline → **C2** linearize → **C3** MoE/ternary →
**C4** mount HRM → **C5** SFT → **C6** self-play RL → **C7** pack → **C8** eval/release.
This notebook runs **C1 (donor eval)** and a generic **LoRA shard trainer** you point at any phase's data.

> ⚠️ Read the review doc `docs/v3/REVIEW_2026-07.md` first — it changes what C2/C3 should be.

## 0. Setup

In [ ]:
# Free-quota friendly. Torch ships with Colab/Kaggle — do not reinstall it.
!pip -q install -U 'transformers>=4.53' 'datasets>=2.20' 'accelerate>=0.33' \n    'peft>=0.12' 'bitsandbytes>=0.43' 'huggingface_hub>=0.24' pyyaml 2>/dev/null
print('deps ready')

## 1. Auth — HF token (never hard-code it)
- **Kaggle**: Add-ons → Secrets → add `HF_TOKEN`.
- **Colab**: 🔑 sidebar → add secret `HF_TOKEN`.
- **Laptop**: `export HF_TOKEN=...` or `huggingface-cli login` once.

In [ ]:
import os
tok = os.environ.get('HF_TOKEN')
if not tok:
    try:  # Kaggle
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
if not tok:
    try:  # Colab
        from google.colab import userdata
        tok = userdata.get('HF_TOKEN')
    except Exception:
        pass
assert tok, 'Set HF_TOKEN (secret/env). It needs WRITE on your checkpoint repo.'
os.environ['HF_TOKEN'] = tok
from huggingface_hub import login; login(tok, add_to_git_credential=False)
print('hf auth ok')

## 2. Platform + GPU

In [ ]:
import torch, platform
def where():
    if os.path.exists('/kaggle'): return 'kaggle'
    if 'COLAB_GPU' in os.environ or os.path.exists('/content'): return 'colab'
    return 'local'
PLATFORM = where()
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU-only'
vram = torch.cuda.get_device_properties(0).total_memory/1e9 if torch.cuda.is_available() else 0
print(f'platform={PLATFORM}  gpu={gpu}  vram={vram:.1f}GB  torch={torch.__version__}')
# Time-box below platform hard limits (leave margin for the final push).
SHARD_MINUTES = {'kaggle': 160, 'colab': 150, 'local': 600}[PLATFORM]

## 3. Get the repo + CheckpointSync
Set `REPO_URL` to your AVA repo (git). We import the tested fabric rather than re-implement it.

In [ ]:
import sys, pathlib
REPO_URL = os.environ.get('AVA_REPO_URL', 'https://github.com/NAME0x0/AVA.git')  # TODO: your remote
ROOT = pathlib.Path('/kaggle/working' if PLATFORM=='kaggle' else '/content' if PLATFORM=='colab' else '.')
repo = ROOT / 'AVA'
if not repo.exists():
    # private repo: embed token in the clone URL for HTTPS auth
    url = REPO_URL.replace('https://', f'https://{tok}@') if 'github' in REPO_URL else REPO_URL
    os.system(f'git clone --depth 1 {url} {repo}')
scripts = repo / 'experiments' / 'exp6_v3' / 'scripts'
assert scripts.exists(), f'not found: {scripts} — fix REPO_URL'
sys.path.insert(0, str(scripts.parent))  # exp6_v3 root -> scripts.*, train.*, engine.*
from scripts.checkpoint_sync import CheckpointSync
print('CheckpointSync imported from', scripts)

## 4. Run config — the only cell you edit per phase

In [ ]:
CKPT_REPO = 'NAME0x0/AVA-v3-checkpoints'   # private Hub repo; auto-created
PHASE     = 'C1'                            # C1 baseline | C2 linearize | C5 sft | ...
DONOR     = 'Qwen/Qwen3.5-4B'               # swapped 2026-07-02 (REVIEW doc section 10)
SEED      = 1234
SEQ_LEN   = 1024
MICRO_BS  = 1
GRAD_ACC  = 16
LR        = 1e-4
SYNC_MIN  = 30                              # push cadence -> max work lost on preempt
TOTAL_STEPS = 20_000                        # phase length in optimizer steps
import random, numpy as np
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 5. C1 — donor baseline (the bar; run once per mode on Kaggle)
Full HumanEval+ / MBPP+ executed in the sandbox + ARC-E/MMLU floors, both
thinking modes. Logic lives in `train/c1_eval.py` (tested repo code). Report
JSON pushed to the checkpoint repo — `train/gate.py` compares every later
phase against it. Skip for training runs.

In [ ]:
if PHASE == 'C1':
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from train.c1_eval import run_c1

    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                             bnb_4bit_compute_dtype=torch.bfloat16,
                             bnb_4bit_use_double_quant=True)
    tok_ = AutoTokenizer.from_pretrained(DONOR, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(DONOR, quantization_config=bnb,
                                                 device_map='auto', trust_remote_code=True,
                                                 torch_dtype=torch.bfloat16)
    report = run_c1(model, tok_, out_path='c1_donor_baseline.json',
                    modes=(False, True),          # non-think first (cheap)
                    code_limit=None,               # full HumanEval+/MBPP+
                    hub_repo=CKPT_REPO)
    for mode, r in report.items():
        if mode != 'meta':
            print(mode, {k: v['score'] for k, v in r.items()})

## 6. Training shard (C5 lane) — config-driven, resumable
All logic in `train/sft.py` + `configs/v30_sft.yaml` (both under test).
Per-source mixture cursor rides inside each checkpoint -> resume anywhere
restores the exact data position. Edit the YAML, not this cell.

In [ ]:
if PHASE != 'C1':
    from train.sft import SFTConfig, train_shard

    cfg = SFTConfig.from_yaml(str(repo / 'experiments/exp6_v3/configs/v30_sft.yaml'))
    cfg.phase = PHASE
    cfg.ckpt_repo = CKPT_REPO
    cfg.donor = DONOR
    cfg.shard_minutes = SHARD_MINUTES
    if PLATFORM == 'local':   # 4 GB VRAM: shorter seqs, more accumulation
        cfg.seq_len, cfg.grad_accum = 640, 32
    summary = train_shard(cfg)
    print('shard summary:', summary)

## 7. Resume anywhere
Nothing to configure. Re-open this notebook on **any** platform, run cells 0–4 and 6,
call `train()`. It pulls `LATEST.json` for `PHASE`, restores trainable weights + optimizer
+ RNG, fast-forwards the stream, and keeps going. Kaggle out of quota → switch to Colab →
switch to laptop overnight. Same repo, same run.

**Artifacts** (all in `NAME0x0/AVA-v3-checkpoints`):
`checkpoints/<phase>/step<N>/state.pt` + `checkpoints/<phase>/LATEST.json`.
At each phase gate, export the winning checkpoint to `NAME0x0/AVA-v3` (BF16) and later a GGUF build.